Bài Tập Về Nhà

In [1]:
# Bài 1. Cài đặt thuật toán AKT vào bài toán 15 puzzle(n > 4)
import copy
from heapq import heappush, heappop

n = 5

rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class HangDoiUuTien:
    def __init__(self):
        self.heap = []

    def push(self, key):
        heappush(self.heap, key)

    def pop(self):
        return heappop(self.heap)

    def empty(self):
        return not self.heap

class TrangThaiNode:
    def __init__(self, node_cha, ma_tran, vi_tri_o_trong, chi_phi_h, so_buoc_g):
        self.node_cha = node_cha
        self.ma_tran = ma_tran
        self.vi_tri_o_trong = vi_tri_o_trong
        self.chi_phi_h = chi_phi_h
        self.so_buoc_g = so_buoc_g

    def __lt__(self, node_ke_tiep):
        return (self.chi_phi_h + self.so_buoc_g) < (node_ke_tiep.chi_phi_h + node_ke_tiep.so_buoc_g)

def tinhChiPhi(ma_tran, ma_tran_dich) -> int:
    dem_sai_vi_tri = 0
    for i in range(n):
        for j in range(n):
            if ma_tran[i][j] != 0 and ma_tran[i][j] != ma_tran_dich[i][j]:
                dem_sai_vi_tri += 1
    return dem_sai_vi_tri

def taoTrangThaiMoi(ma_tran, vi_tri_o_trong, vi_tri_moi, so_buoc_g, node_cha, ma_tran_dich) -> TrangThaiNode:
    ma_tran_moi = copy.deepcopy(ma_tran)

    x1, y1 = vi_tri_o_trong
    x2, y2 = vi_tri_moi

    ma_tran_moi[x1][y1], ma_tran_moi[x2][y2] = ma_tran_moi[x2][y2], ma_tran_moi[x1][y1]

    chi_phi_h = tinhChiPhi(ma_tran_moi, ma_tran_dich)
    node_moi = TrangThaiNode(node_cha, ma_tran_moi, vi_tri_moi, chi_phi_h, so_buoc_g)
    return node_moi

def inMaTran(ma_tran):
    for i in range(n):
        for j in range(n):
            print("%2d " % (ma_tran[i][j]), end=" ")
        print()

def kiemTraHopLe(x, y):
    return 0 <= x < n and 0 <= y < n

def inDuongDi(root):
    if root is None:
        return

    inDuongDi(root.node_cha)
    inMaTran(root.ma_tran)
    print("-------")

def giaiQuyet(ma_tran_dau, vi_tri_o_trong, ma_tran_dich):
    pq = HangDoiUuTien()

    chi_phi_h = tinhChiPhi(ma_tran_dau, ma_tran_dich)
    root = TrangThaiNode(None, ma_tran_dau, vi_tri_o_trong, chi_phi_h, 0)

    pq.push(root)

    da_duyet = set()

    while not pq.empty():
        node_hien_tai = pq.pop()

        trang_thai_tupple = tuple(tuple(row) for row in node_hien_tai.ma_tran)
        if trang_thai_tupple in da_duyet:
            continue
        da_duyet.add(trang_thai_tupple)

        if node_hien_tai.chi_phi_h == 0:
            print("ĐÃ TÌM THẤY ĐƯỜNG ĐI:")
            inDuongDi(node_hien_tai)
            return

        for i in range(4):
            vi_tri_moi = [
                node_hien_tai.vi_tri_o_trong[0] + rows[i],
                node_hien_tai.vi_tri_o_trong[1] + cols[i],
            ]
            if kiemTraHopLe(vi_tri_moi[0], vi_tri_moi[1]):
                child = taoTrangThaiMoi(
                    node_hien_tai.ma_tran,
                    node_hien_tai.vi_tri_o_trong,
                    vi_tri_moi,
                    node_hien_tai.so_buoc_g + 1,
                    node_hien_tai,
                    ma_tran_dich
                )
                pq.push(child)

ma_tran_dich = [
    [ 1,  2,  3,  4,  5],
    [ 6,  7,  8,  9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24,  0]
]

ma_tran_dau = [
    [ 1,  2,  3,  4,  5],
    [ 6,  7,  8,  9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18,  0, 20],
    [21, 22, 23, 19, 24]
]

vi_tri_o_trong = [3, 3]

giaiQuyet(ma_tran_dau, vi_tri_o_trong, ma_tran_dich)

ĐÃ TÌM THẤY ĐƯỜNG ĐI:
 1   2   3   4   5  
 6   7   8   9  10  
11  12  13  14  15  
16  17  18   0  20  
21  22  23  19  24  
-------
 1   2   3   4   5  
 6   7   8   9  10  
11  12  13  14  15  
16  17  18  19  20  
21  22  23   0  24  
-------
 1   2   3   4   5  
 6   7   8   9  10  
11  12  13  14  15  
16  17  18  19  20  
21  22  23  24   0  
-------


In [9]:
#Bài 2. Cài đặt bài toán người giao hàng theo thuật toán A*
from collections import deque

class Graph:
    def __init__(self, adjac_lis, heuristic_vals):
        self.adjac_lis = adjac_lis
        self.H = heuristic_vals

    def get_neighbors(self, v):
        return self.adjac_lis.get(v, [])

    def h(self, n):
        return self.H.get(n, 1)

    def a_star_algorithm(self, start, stop):
        danh_sach_dinh = set(self.adjac_lis.keys())
        for d in self.adjac_lis.values():
            for ke, _ in d:
                danh_sach_dinh.add(ke)

        if start not in danh_sach_dinh or stop not in danh_sach_dinh:
            print(f"Địa điểm '{start}' hoặc '{stop}' không tồn tại trên bản đồ giao hàng!")
            return None

        open_lst = set([start])
        closed_lst = set([])

        g = {}
        g[start] = 0

        parents = {}
        parents[start] = start

        while len(open_lst) > 0:
            n = None

            for v in open_lst:
                if n == None or g[v] + self.h(v) < g[n] + self.h(n):
                    n = v

            if n == None:
                print('Không thể tìm thấy tuyến đường giao hàng khả thi!')
                return None

            if n == stop:
                reconst_path = []
                while parents[n] != n:
                    reconst_path.append(n)
                    n = parents[n]
                reconst_path.append(start)
                reconst_path.reverse()

                print(f"Tuyến đường giao hàng tối ưu từ trạm {start} đến trạm {stop} là: {' ➔ '.join(reconst_path)}")
                print(f"Tổng khoảng cách / chi phí vận chuyển: {g[stop]}")
                return reconst_path

            # Duyệt qua các đỉnh kề của n
            for (m, weight) in self.get_neighbors(n):
                if m not in open_lst and m not in closed_lst:
                    open_lst.add(m)
                    parents[m] = n
                    g[m] = g[n] + weight
                else:
                    if g[m] > g[n] + weight:
                        g[m] = g[n] + weight
                        parents[m] = n
                        if m in closed_lst:
                            closed_lst.remove(m)
                            open_lst.add(m)

            open_lst.remove(n)
            closed_lst.add(n)

        print('Không thể tìm thấy tuyến đường giao hàng khả thi!')
        return None

adjac_lis = {
    'A': [('B', 10), ('C', 3)],
    'B': [('C', 4), ('D', 5)],
    'C': [('D', 12), ('A', 7)]
}

heuristic_vals = {
    'A': 1,
    'B': 2,
    'C': 3,
    'D': 4
}

# Khởi tạo Graph
graph1 = Graph(adjac_lis, heuristic_vals)

dinh_bat_dau = 'A'
dinh_ket_thuc = 'D'

print("--- HỆ THỐNG ĐIỀU PHỐI ĐƯỜNG ĐI CHO NGƯỜI GIAO HÀNG ---")
graph1.a_star_algorithm(dinh_bat_dau, dinh_ket_thuc)

--- HỆ THỐNG ĐIỀU PHỐI ĐƯỜNG ĐI CHO NGƯỜI GIAO HÀNG ---
Tuyến đường giao hàng tối ưu từ trạm A đến trạm D là: A ➔ C ➔ D
Tổng khoảng cách / chi phí vận chuyển: 15


['A', 'C', 'D']